# ML-04 — Search Intelligence Data Contract

This notebook contains the data contract and verification queries for the **Refresh / Content Opportunity Scoring** lane (Lane 2) using the Hugging Face warehouse dataset.

## 1. Unit of analysis + time window

### Plain-Words Contract Answers:
1. **Unit of Analysis:** One row = one unique content item (`content_hash_id`) for a specific client (`client_hash_id`) aggregated over the month.
2. **Tables Used:** `dim_content` and `fact_content_daily_performance` (referenced dynamically inside DuckDB).
3. **Time Window:** March 2026 (`month = '2026-03'`). 
   * *Feature Window:* March 1, 2026 to March 15, 2026.
   * *Target Window:* March 16, 2026 to March 31, 2026.
4. **Target / Proxy to Predict:** Traffic decline (`is_declining = 1` if search impressions in the target window are less than 80% of impressions in the feature window).
5. **Deliberate Exclusion:** FlyRank's internal product decision flags and priority scores (e.g., `health_score`, `priority_score`, `action_type`), to prevent circular logic and ensure we model only raw observed traffic patterns.

In [ ]:
import os
import duckdb
import pandas as pd

# Load Hugging Face token from local .env file
hf_token = None
if os.path.exists("../../.env"):
    with open("../../.env", "r") as f:
        for line in f:
            if line.strip().startswith("HF_TOKEN"):
                parts = line.split("=", 1)
                if len(parts) == 2:
                    hf_token = parts[1].strip().strip('"').strip("'")

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

print("DuckDB connected and authenticated successfully!")

## 2. Fields: feature / label / context / excluded

* **Features:**
  * `imp_past` (Sum of `gsc_impressions` during March 1–15)
  * `clk_past` (Sum of `gsc_clicks` during March 1–15)
  * `pos_past` (Average `gsc_avg_position` during March 1–15)
  * `ses_past` (Sum of `ga4_sessions` during March 1–15)
  * `scroll_past` (Sum of `scroll_events` during March 1–15)
* **Label / Proxy:**
  * `is_declining` (1 if `imp_future` < 0.8 * `imp_past`, 0 otherwise)
* **Context:**
  * `client_hash_id` (used for grouping / cross-validation)
  * `content_hash_id` (uniquely identifies the page)
* **Excluded:**
  * `imp_future` (future impressions; used to compute label, must be excluded from features to prevent leakage)
  * All internal decision rules/heuristics from the system (not present in raw facts, but excluded by design)

In [ ]:
# Verify field categorization strategy
feature_cols = ['imp_past', 'clk_past', 'pos_past', 'ses_past', 'scroll_past']
context_cols = ['client_hash_id', 'content_hash_id']
label_col = 'is_declining'
print(f"Staging {len(feature_cols)} features and {len(context_cols)} context columns.")

## 3. Verify it with queries (grain, counts, missing values, windows)

We run three verification queries on the `month = '2026-03'` partition to prove our contract claims.

In [ ]:
# Query 1: Verify the Grain (One row really is unique content per client)
q1 = f"""
SELECT client_hash_id, content_hash_id, COUNT(*) c
FROM (
    SELECT client_hash_id, content_hash_id
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2
)
GROUP BY 1, 2
HAVING c > 1
LIMIT 5
"""
print("Query 1 (Uniqueness check, empty DataFrame means success):")
print(con.sql(q1).df())

In [ ]:
# Query 2: Slice's Row Count and Date Span
q2 = f"""
SELECT COUNT(DISTINCT content_hash_id) AS unique_pages,
       COUNT(*) AS total_daily_rows,
       MIN(report_date) AS min_date,
       MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""
print("Query 2 (Total size and date span for March 2026):")
print(con.sql(q2).df())

In [ ]:
# Query 3: Availability (Filter with IS TRUE and show how many survive)
q3 = f"""
SELECT COUNT(*) AS total_daily_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_avail_rows,
       SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_avail_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS both_avail_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""
print("Query 3 (Availability numbers for GA4 and GSC columns):")
print(con.sql(q3).df())

## 4. Feature Extraction & The Leakage Trap

First, we build our feature frame for `month = '2026-03'` using DuckDB. We filter to pages that had at least 50 impressions in the first half of the month to keep the dataset clean.

### Five Features and Availability Rationale:
1. `imp_past`: **Knowable at decision moment** because search visibility is logged historical metadata prior to March 15.
2. `clk_past`: **Knowable at decision moment** because search clicks are logged historical metadata prior to March 15.
3. `pos_past`: **Knowable at decision moment** because search positions are logged historical metadata prior to March 15.
4. `ses_past`: **Knowable at decision moment** because user sessions are logged historical metadata prior to March 15.
5. `scroll_past`: **Knowable at decision moment** because scroll events are logged historical metadata prior to March 15.

In [ ]:
df_features = con.sql(f"""
    WITH monthly_data AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            -- Features: aggregated over March 1 to March 15
            SUM(CASE WHEN report_date BETWEEN '2026-03-01' AND '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_past,
            SUM(CASE WHEN report_date BETWEEN '2026-03-01' AND '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_past,
            AVG(CASE WHEN report_date BETWEEN '2026-03-01' AND '2026-03-15' THEN gsc_avg_position ELSE NULL END) AS pos_past,
            SUM(CASE WHEN report_date BETWEEN '2026-03-01' AND '2026-03-15' THEN ga4_sessions ELSE 0 END) AS ses_past,
            SUM(CASE WHEN report_date BETWEEN '2026-03-01' AND '2026-03-15' THEN scroll_events ELSE 0 END) AS scroll_past,
            
            -- Target components
            SUM(CASE WHEN report_date BETWEEN '2026-03-16' AND '2026-03-31' THEN gsc_impressions ELSE 0 END) AS imp_future
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
        HAVING imp_past >= 50
    )
    SELECT 
        *,
        CASE WHEN imp_future < 0.8 * imp_past THEN 1 ELSE 0 END AS is_declining
    FROM monthly_data
""").df()

print(f"Extracted feature frame with {len(df_features):,} rows.")
df_features.head()

### Performing the Leakage Experiment:

We will intentionally add `imp_future` (a target-derived column) into our feature list and train a Random Forest model. We will observe the test score jump to near-perfect, then remove the leaked column and run the honest model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Handle any missing values in position by filling with a default large position value
df_features['pos_past'] = df_features['pos_past'].fillna(100.0)

# Define targets and feature split
y = df_features['is_declining']

print("--- LEAKED EXPERIMENT (using imp_future as a feature) ---")
X_leaked = df_features[['imp_past', 'clk_past', 'pos_past', 'ses_past', 'scroll_past', 'imp_future']]
X_tr_leak, X_te_leak, y_tr, y_te = train_test_split(X_leaked, y, test_size=0.3, random_state=42, stratify=y)

model_leak = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model_leak.fit(X_tr_leak, y_tr)
preds_leak = model_leak.predict(X_te_leak)
print(f"Leaked model accuracy: {accuracy_score(y_te, preds_leak):.4f}")
print(classification_report(y_te, preds_leak))

In [ ]:
print("--- HONEST EXPERIMENT (removing imp_future from features) ---")
X_honest = df_features[['imp_past', 'clk_past', 'pos_past', 'ses_past', 'scroll_past']]
X_tr_honest, X_te_honest, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)

model_honest = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model_honest.fit(X_tr_honest, y_tr)
preds_honest = model_honest.predict(X_te_honest)
print(f"Honest model accuracy: {accuracy_score(y_te, preds_honest):.4f}")
print(classification_report(y_te, preds_honest))

## 5. Data limits

### One Named Limitation of This Slice:
**Unbalanced Panel Tracking:** In the daily performance table, different clients have completely different history start dates. For instance, some clients only have GA4 tracking starting mid-2025, which means early rows for those clients have GA4 metrics set to zero (`ga4_data_available = FALSE`). If we treat missing tracking data as zero traffic, we introduce severe bias. Models must filter or adjust for `ga4_data_available` to keep analyses honest.